In [1]:
import os
import sys
import threading
from datetime import datetime

sys.path.append("/data/alop/eye_transformer/")
from tokenizer_aligner.tokenizer_aligner import TokenizerAligner
from transformers import AutoTokenizer

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorWithPadding,
    BatchEncoding,
)
import re
from functools import partial
from datasets import load_dataset, Dataset

/home/alop/miniconda3/envs/cache_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
tokenizer_name = "t5-small"
tokenizer_fix = AutoTokenizer.from_pretrained(
    tokenizer_name, cache_dir="./cache/models", model_max_length=2048
)
# tokenizer = T5Tokenizer.from_pretrained('t5-small')
tokenizer_name = "meta-llama/Meta-Llama-3-8B"
tokenizer_model = AutoTokenizer.from_pretrained(tokenizer_name)
tokenizer_model.add_special_tokens({"pad_token": "[PAD]"})
dataset = "timdettmers/openassistant-guanaco"
data = load_dataset(dataset, split="train[:1%]")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Repo card metadata block was not found. Setting CardData to empty.


Miro todos los que tienen error cuando se intentan cruzar.

In [6]:
for num, text_test in enumerate(data["text"]):
    # text_test = ["hello I am desperated"]
    try:
        text_tokenized_model = tokenizer_model(
            [text_test], padding=True, truncation=True, add_special_tokens=True
        )
        texts = []
        for i in range(len(text_tokenized_model["input_ids"])):
            texts.append(
                tokenizer_model.decode(
                    text_tokenized_model["input_ids"][i], skip_special_tokens=True
                )
            )
        text_tokenized_fix = tokenizer_fix(
            texts, padding=True, truncation=True, add_special_tokens=True
        )
        # --------------------------------------------------------------------------------------------
        token_idx_mapped, words_str_mapped = TokenizerAligner().align_tokens(
            texts, text_tokenized_model, text_tokenized_fix, return_words=True
        )
        token_idx_str_mapped = TokenizerAligner().map_tokens_to_str(
            token_idx_mapped,
            text_tokenized_model,
            text_tokenized_fix,
        )
        # --------------------------------------------------------------------------------------------
    except Exception as e:
        print(num, ":", e)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


14 : Error in the alignment of the tokens
44 : Error in the alignment of the tokens
50 : Error in the alignment of the tokens
57 : Error in the alignment of the tokens
65 : Error in the alignment of the tokens
71 : Error in the alignment of the tokens
90 : Error in the alignment of the tokens


Mirare en el primer ejemplo por que funciona o no funciona.

In [7]:
def _map_words(i, j, comb_sorted, main_list, compare_list, result_ids, result_words):
    asigned = 0
    for k, l in comb_sorted:
        if "".join(main_list[i : i + l]) == "".join(compare_list[j : j + k]):
            result_ids.append((list(range(i, i + l)), list(range(j, j + k))))
            result_words.append(
                (
                    "".join(main_list[i : i + l]),
                    "".join(compare_list[j : j + k]),
                )
            )
            i += l
            j += k
            asigned = 1
            break
    return asigned, i, j, result_ids, result_words


def map_words(main_list_prev: list, compare_list_prev: list):
    result_ids, result_words = [], []
    i, j = 0, 0
    main_list = [re.sub(r"\s+", "", word) for word in main_list_prev]
    compare_list = [re.sub(r"\s+", "", word) for word in compare_list_prev]
    # we sorted the combinations to prioritize the most probables
    comb = [(i, j) for j in range(1, 15) for i in range(1, 11)]
    comb_sorted = sorted(comb, key=lambda x: x[0] + x[1], reverse=False)
    while i < len(main_list) and j < len(compare_list):
        asigned, i, j, result_ids, result_words = _map_words(
            i, j, comb_sorted, main_list, compare_list, result_ids, result_words
        )
        # print(asigned, i, j)
        if asigned == 1:
            continue
        else:
            print
            return i, j
            break

    return result_ids, result_words

In [8]:
text_test = data["text"][14]

text_tokenized_model = tokenizer_model(
    text_test, padding=True, truncation=True, add_special_tokens=True
)
texts = []
if isinstance(text_tokenized_model["input_ids"][0], list):
    for i in range(len(text_tokenized_model["input_ids"])):
        texts.append(
            tokenizer_model.decode(
                text_tokenized_model["input_ids"][i], skip_special_tokens=True
            )
        )
else:
    texts = tokenizer_model.decode(
        text_tokenized_model["input_ids"], skip_special_tokens=True
    )

text_tokenized_fix = tokenizer_fix(
    texts, padding=True, truncation=True, add_special_tokens=True
)

word_token_idx_first = text_tokenized_model.word_ids()
word_token_idx_second = text_tokenized_fix.word_ids()

words_first = TokenizerAligner.text_to_words(
    texts, text_tokenized_model, word_token_idx_first
)
words_second = TokenizerAligner.text_to_words(
    texts, text_tokenized_fix, word_token_idx_second
)

In [9]:
i, j = map_words(words_first, words_second)
print(i, words_first[i], j, words_second[j])
print(i, words_first[i : i + 6], j, words_second[j : j + 6])

335 s 187 status
335 ['s', 'tatus o', 'f 4', '0', '0 b', 'ad r'] 187 ['status', 'of', '400', 'bad', 'request', 'instead']


In [10]:
print(texts)

### Human: I'm creating a NodeJS backend using Express, and I'm getting the following error:
`Uncaught TypeError: Cannot read properties of undefined (reading 'id')`

My code of where this error is happening is as follows:
```
app.delete('/post', (req, res) => {
    const user = getUser(req.user.user_id);
    if (!user) return res.sendStatus(401);
    const post = getPost(req.body.id);
    if (!post) return res.sendStatus(404);
    if (post.user_id!= req.user.user_id) return res.sendStatus(403);
    // Code that deletes the post through DB
    return res.sendStatus(201);
});
```

What are ways I can fix this error?### Assistant: The error "Uncaught TypeError: Cannot read properties of undefined (reading 'id')" is likely occurring because "req.body.id" is undefined. To resolve this error, you should add a check to verify that "req.body.id" exists before trying to access its properties.

You can do this by adding a conditional statement before accessing "req.body.id":

'''
app.delete('/p

Lo que parece aqui es que las palabras del primer tokenizer de llama no estan bien cuadradas. 
como si realmente cuando he calculado las palabras con el primer tokenizer los indices de los caracteres no se hubieran calculado bien. Por ello exploro como se crean esos caracteres. 

In [11]:
text_test = data["text"][14]
text_test = re.sub(r" !", "!", text_test)
# text_test = re.sub(r"[\'`]" , '', text_test)
# text_test = re.sub(r'`', '', text_test)
# text_test = text_test[400:500]
# text_test = 'st) return res.sendStatus(404);\n    if (post.user_id != req.user.user_id) return res.sendStatus(403)'
# text_test = '(post.user_id != req.user.user_id)'
# text_test = re.sub(r' !', '!', text_test)
# text_test = """st) return res.sendStatus(404);
#     if (post.user_id!= req.user.user_id) return res.sendStatus(403);
#     // Code that deletes the post through DB"""
# print(text_test)
# text_test = ["hello I am desperated"]
text_tokenized_model = tokenizer_model(
    text_test, padding=True, truncation=True, add_special_tokens=True
)
texts = []
if isinstance(text_tokenized_model["input_ids"][0], list):
    for i in range(len(text_tokenized_model["input_ids"])):
        texts.append(
            tokenizer_model.decode(
                text_tokenized_model["input_ids"][i], skip_special_tokens=True
            )
        )
else:
    texts = tokenizer_model.decode(
        text_tokenized_model["input_ids"], skip_special_tokens=True
    )

text_tokenized_fix = tokenizer_fix(
    texts, padding=True, truncation=True, add_special_tokens=True
)

word_token_idx_first = text_tokenized_model.word_ids()
word_token_idx_second = text_tokenized_fix.word_ids()

words_first = TokenizerAligner.text_to_words(
    texts, text_tokenized_model, word_token_idx_first
)
words_second = TokenizerAligner.text_to_words(
    texts, text_tokenized_fix, word_token_idx_second
)
word_tokens_model = TokenizerAligner().words_tokens_to_str(
    word_token_idx_first, text_tokenized_model, tokenizer_model
)

In [12]:
def text_to_words(text, text_tokenized, word_token_ids: list):
    words = []
    # print(text)
    for i in range(1 + max([x if x is not None else 0 for x in word_token_ids])):
        start, end = text_tokenized.word_to_chars(i)
        # print(i, start, end, text[start:end].strip().lower(), text[start-1:end-1].strip().lower())
        words.append(text[start:end].strip().lower())
    return words

In [13]:
words_model = text_to_words(texts, text_tokenized_model, word_token_idx_first)

In [14]:
len(word_tokens_model)
for i in range(len(words_model)):
    print(i, words_model[i], "-----", word_tokens_model[i])

0 ### ----- (['###'],)
1 human ----- (['ĠHuman'],)
2 : ----- ([':'],)
3 i ----- (['ĠI'],)
4 'm ----- (["'m"],)
5 creating ----- (['Ġcreating'],)
6 a ----- (['Ġa'],)
7 nodejs ----- (['ĠNode', 'JS'],)
8 backend ----- (['Ġbackend'],)
9 using ----- (['Ġusing'],)
10 express ----- (['ĠExpress'],)
11 , ----- ([','],)
12 and ----- (['Ġand'],)
13 i ----- (['ĠI'],)
14 'm ----- (["'m"],)
15 getting ----- (['Ġgetting'],)
16 the ----- (['Ġthe'],)
17 following ----- (['Ġfollowing'],)
18 error ----- (['Ġerror'],)
19 : ----- ([':Ċ'],)
20 `uncaught ----- (['`', 'Un', 'caught'],)
21 typeerror ----- (['ĠTypeError'],)
22 : ----- ([':'],)
23 cannot ----- (['ĠCannot'],)
24 read ----- (['Ġread'],)
25 properties ----- (['Ġproperties'],)
26 of ----- (['Ġof'],)
27 undefined ----- (['Ġundefined'],)
28 ( ----- (['Ġ('],)
29 reading ----- (['reading'],)
30 ' ----- (["Ġ'"],)
31 id ----- (['id'],)
32 ')` ----- (["')", '`ĊĊ'],)
33 my ----- (['My'],)
34 code ----- (['Ġcode'],)
35 of ----- (['Ġof'],)
36 where ----- (['Ġ

Despues de muchas pruebas comparando las palabras entre tokenizers y posteriormente las palabras del tokenizer de llama con los tokens que se supone que tiene que tener, he encontrado que el problema es que el primer tokenizer no esta calculando bien los indices de los caracteres cuando hay una exclamacion
en el ejemplo 14 se arregla con :
text_test = re.sub(r' !', '!', text_test)

In [22]:
for num, text_test in enumerate(data["text"]):
    # text_test = re.sub(r' !', '!', text_test)
    text_test = re.sub(r"\s([.,;:!?¿¡])", r"\1", text_test)
    try:
        text_tokenized_model = tokenizer_model(
            [text_test], padding=True, truncation=True, add_special_tokens=True
        )
        texts = []
        for i in range(len(text_tokenized_model["input_ids"])):
            texts.append(
                tokenizer_model.decode(
                    text_tokenized_model["input_ids"][i], skip_special_tokens=True
                )
            )
        text_tokenized_fix = tokenizer_fix(
            texts, padding=True, truncation=True, add_special_tokens=True
        )
        # --------------------------------------------------------------------------------------------
        token_idx_mapped, words_str_mapped = TokenizerAligner().align_tokens(
            texts, text_tokenized_model, text_tokenized_fix, return_words=True
        )
        token_idx_str_mapped = TokenizerAligner().map_tokens_to_str(
            token_idx_mapped,
            text_tokenized_model,
            text_tokenized_fix,
        )
        # --------------------------------------------------------------------------------------------
    except Exception as e:
        print("-------------------------")
        print(num, ":", e)
        print(text_test[:222])

-------------------------
50 : Error in the alignment of the tokens
### Human: Плюсы и минусы языков программирования C++ и Rust. Что лучше?### Assistant: В зависимости от задач и проекта выбор языка программирования может варьироваться. Ответа, подходящего под любые случаи нет.
Оба языка 
-------------------------
57 : Error in the alignment of the tokens
### Human: 布洛芬和对乙酰氨基酚的区别是？### Assistant: 根据维基百科，
布洛芬（英语：Ibuprofen），是一种非类固醇消炎药（NSAID），常用来止痛、退烧、消炎。可用于治疗经痛、偏头痛，和类风湿性关节炎。
对乙酰氨基酚（英语：Acetaminophen、Paracetamol、简称：APAP），又称乙酰胺酚、扑热息痛，是一种用于治疗疼痛与发烧的药物。
这是两种不同的药，详情可以参考维基百科.
参考链接：
布洛
-------------------------
65 : Error in the alignment of the tokens
### Human: 请解释一下颜色空间BT601和颜色空间BT709的区别。### Assistant: BT601和BT709是两种视频颜色空间标准，它们都规定了视频信号在颜色表示上使用的色彩范围和色度。它们的主要区别在于，BT601是旧版标准，而BT709是更现代的标准。

具体来说，BT601是一种标准的视频颜色空间，它规定了标准定义视频的亮度、色度、色彩饱和度等特性。BT601旨在标准化模拟电视信号的颜色空间，通常用于标准的480
-------------------------
90 : Error in the alignment of the tokens
### Human: How can I get an EIN for my new business?### Ass

In [25]:
text_test = data["text"][90]
text_test = re.sub(r" !", "!", text_test)

text_tokenized_model = tokenizer_model(
    text_test, padding=True, truncation=True, add_special_tokens=True
)
texts = []
if isinstance(text_tokenized_model["input_ids"][0], list):
    for i in range(len(text_tokenized_model["input_ids"])):
        texts.append(
            tokenizer_model.decode(
                text_tokenized_model["input_ids"][i], skip_special_tokens=True
            )
        )
else:
    texts = tokenizer_model.decode(
        text_tokenized_model["input_ids"], skip_special_tokens=True
    )

text_tokenized_fix = tokenizer_fix(
    texts, padding=True, truncation=True, add_special_tokens=True
)

word_token_idx_first = text_tokenized_model.word_ids()
word_token_idx_second = text_tokenized_fix.word_ids()

words_first = TokenizerAligner.text_to_words(
    texts, text_tokenized_model, word_token_idx_first
)
words_second = TokenizerAligner.text_to_words(
    texts, text_tokenized_fix, word_token_idx_second
)
word_tokens_model = TokenizerAligner().words_tokens_to_str(
    word_token_idx_first, text_tokenized_model, tokenizer_model
)
# for i in range(len(words_model)):
#     print(i, words_first[i],'-----', word_tokens_model[i])
word_tokens_fix = TokenizerAligner().words_tokens_to_str(
    word_token_idx_second, text_tokenized_fix, tokenizer_fix
)
for i in range(len(words_second)):
    print(i, words_second[i], "-----", word_tokens_fix[i])

0 ### ----- (['▁#', '##'],)
1 human: ----- (['▁Human', ':'],)
2 how ----- (['▁How'],)
3 can ----- (['▁can'],)
4 i ----- (['▁I'],)
5 get ----- (['▁get'],)
6 an ----- (['▁an'],)
7 ein ----- (['▁E', 'IN'],)
8 for ----- (['▁for'],)
9 my ----- (['▁my'],)
10 new ----- (['▁new'],)
11 business?### ----- (['▁business', '?', '#', '##'],)
12 assistant: ----- (['▁Assistant', ':'],)
13 an ----- (['▁An'],)
14 employer ----- (['▁Employ', 'er'],)
15 identification ----- (['▁Identification'],)
16 number ----- (['▁Number'],)
17 (ein) ----- (['▁(', 'E', 'IN', ')'],)
18 is ----- (['▁is'],)
19 a ----- (['▁', 'a'],)
20 unique ----- (['▁unique'],)
21 nine-digit ----- (['▁nine', '-', 'digit'],)
22 number ----- (['▁number'],)
23 assigned ----- (['▁assigned'],)
24 by ----- (['▁by'],)
25 the ----- (['▁the'],)
26 internal ----- (['▁Internal'],)
27 revenue ----- (['▁Revenue'],)
28 service ----- (['▁Service'],)
29 (irs) ----- (['▁(', 'IR', 'S', ')'],)
30 to ----- (['▁to'],)
31 businesses ----- (['▁businesses'],)
32

In [26]:
map_words(words_first, words_second)

373 it 128 https://www.irs.gov/businesses/small-businesses-self-employed/apply-for-an-employer-identification-number-ein-online
373 ['it', 'to', ':', 'internal', 'revenue', 'service'] 128 ['https://www.irs.gov/businesses/small-businesses-self-employed/apply-for-an-employer-identification-number-ein-online', '3.', 'complete', 'the', 'application:', 'fill']


No ha funcionado el mapeo entre palabras por que una pagina web muy larga uno lo ha dividido en muchas palabras y el otro en una sola. Por lo que cambiare el 

In [18]:
"".join(words_first[i : i + 15]) in "".join(words_second[j])

True

In [19]:
"".join(words_first[i : i + 15])

'https://www.irs.gov/businesses/small-businesses-self-employed/apply-for-an-employer-identification'

In [20]:
"".join(words_second[j])

'https://www.irs.gov/businesses/small-businesses-self-employed/apply-for-an-employer-identification-number-ein-online'

In [6]:
text_test = data["text"][90]
text_test = re.sub(r" !", "!", text_test)

text_tokenized_model = tokenizer_model(
    text_test, padding=True, truncation=True, add_special_tokens=True
)
texts = []
if isinstance(text_tokenized_model["input_ids"][0], list):
    for i in range(len(text_tokenized_model["input_ids"])):
        texts.append(
            tokenizer_model.decode(
                text_tokenized_model["input_ids"][i], skip_special_tokens=True
            )
        )
else:
    texts = tokenizer_model.decode(
        text_tokenized_model["input_ids"], skip_special_tokens=True
    )

text_tokenized_fix = tokenizer_fix(
    texts, padding=True, truncation=True, add_special_tokens=True
)

word_token_idx_first = text_tokenized_model.word_ids()
word_token_idx_second = text_tokenized_fix.word_ids()

words_first = TokenizerAligner.text_to_words(
    texts, text_tokenized_model, word_token_idx_first
)
words_second = TokenizerAligner.text_to_words(
    texts, text_tokenized_fix, word_token_idx_second
)
word_tokens_model = TokenizerAligner().words_tokens_to_str(
    word_token_idx_first, text_tokenized_model, tokenizer_model
)
# for i in range(len(words_model)):
#     print(i, words_first[i],'-----', word_tokens_model[i])
word_tokens_fix = TokenizerAligner().words_tokens_to_str(
    word_token_idx_second, text_tokenized_fix, tokenizer_fix
)
for i in range(len(words_second)):
    print(i, words_second[i], "-----", word_tokens_fix[i])

token_idx_mapped, words_str_mapped = TokenizerAligner().align_tokens(
    texts, text_tokenized_model, text_tokenized_fix, return_words=True
)

0 ### ----- (['▁#', '##'],)
1 human: ----- (['▁Human', ':'],)
2 how ----- (['▁How'],)
3 can ----- (['▁can'],)
4 i ----- (['▁I'],)
5 get ----- (['▁get'],)
6 an ----- (['▁an'],)
7 ein ----- (['▁E', 'IN'],)
8 for ----- (['▁for'],)
9 my ----- (['▁my'],)
10 new ----- (['▁new'],)
11 business?### ----- (['▁business', '?', '#', '##'],)
12 assistant: ----- (['▁Assistant', ':'],)
13 an ----- (['▁An'],)
14 employer ----- (['▁Employ', 'er'],)
15 identification ----- (['▁Identification'],)
16 number ----- (['▁Number'],)
17 (ein) ----- (['▁(', 'E', 'IN', ')'],)
18 is ----- (['▁is'],)
19 a ----- (['▁', 'a'],)
20 unique ----- (['▁unique'],)
21 nine-digit ----- (['▁nine', '-', 'digit'],)
22 number ----- (['▁number'],)
23 assigned ----- (['▁assigned'],)
24 by ----- (['▁by'],)
25 the ----- (['▁the'],)
26 internal ----- (['▁Internal'],)
27 revenue ----- (['▁Revenue'],)
28 service ----- (['▁Service'],)
29 (irs) ----- (['▁(', 'IR', 'S', ')'],)
30 to ----- (['▁to'],)
31 businesses ----- (['▁businesses'],)
32

In [7]:
for pair in words_str_mapped:
    print(pair)

('###', '###')
('human:', 'human:')
('how', 'how')
('can', 'can')
('i', 'i')
('get', 'get')
('an', 'an')
('ein', 'ein')
('for', 'for')
('my', 'my')
('new', 'new')
('business?###', 'business?###')
('assistant:', 'assistant:')
('an', 'an')
('employer', 'employer')
('identification', 'identification')
('number', 'number')
('(ein)', '(ein)')
('is', 'is')
('a', 'a')
('unique', 'unique')
('nine-digit', 'nine-digit')
('number', 'number')
('assigned', 'assigned')
('by', 'by')
('the', 'the')
('internal', 'internal')
('revenue', 'revenue')
('service', 'service')
('(irs)', '(irs)')
('to', 'to')
('businesses', 'businesses')
('in', 'in')
('the', 'the')
('united', 'united')
('states', 'states')
('for', 'for')
('tax', 'tax')
('purposes.', 'purposes.')
('to', 'to')
('obtain', 'obtain')
('an', 'an')
('ein', 'ein')
('for', 'for')
('your', 'your')
('new', 'new')
('business,', 'business,')
('follow', 'follow')
('these', 'these')
('steps:', 'steps:')
('1.', '1.')
('determine', 'determine')
('eligibility:',

Solucionado el problema con el 90 con la nueva fucniona de clase de tokenizer de map_words_v2